# Ejemplo de Estrategia Swap Spread a 5Y

In [ ]:
from collections import namedtuple
import pandas as pd

import qcfinancial as qcf

import aux_functions as aux
import qcf_wrappers as qcw

In [ ]:
%load_ext autoreload
%autoreload 2

## Carga de Curvas de Mercado

In [ ]:
df_cicpclp = pd.read_excel("input/20250926_cicpclp.xlsx")
df_cclpcolusd = pd.read_csv("input/20250926_cclpcolusd.csv")
df_cclfcolusd = pd.read_excel("input/20250926_cclfcolusd.xlsx")

Se construyen los objetos `ZeroCouponCurve` de `qcfinancial`.

In [ ]:
qcf_cicpclp = aux.build_zero_coupon_curve(
    df_cicpclp.maturity.values,
    df_cicpclp.rate.values,
    cyf := qcw.YearFraction.ACT365,
    cwf := qcw.WealthFactor.CON,
)

In [ ]:
qcf_cclpcolusd = aux.build_zero_coupon_curve(
    df_cclpcolusd.maturity.values,
    df_cclpcolusd.rate.values,
    cyf,
    cwf,
)

In [ ]:
qcf_cclfcolusd = aux.build_zero_coupon_curve(
    df_cclfcolusd.maturity.values,
    df_cclfcolusd.rate.values,
    cyf,
    cwf,
)

In [ ]:
notional_uf = 12_500_000.0

## BTU a 5Y

In [ ]:
scl = aux.get_business_calendar("CL", range(2018, 2031))

In [ ]:
params_bono = dict(
    rec_pay = qcf.RecPay.RECEIVE,
    start_date = qcf.QCDate(1, 3, 2018),
    end_date = qcf.QCDate(1, 9, 2030),
    bus_adj_rule = qcf.BusyAdjRules.NO,
    settlement_periodicity = qcf.Tenor('6M'),
    settlement_stub_period = qcf.StubPeriod.NO,
    settlement_calendar = scl,
    settlement_lag = 0,
    initial_notional = notional_uf,
    amort_is_cashflow = True,
    interest_rate = qcf.QCInterestRate(
        0.019,
        qcf.QC30360(),
        qcf.QCLinearWf(),
    ),
    notional_currency = qcf.QCCLF(),
    is_bond = True,
    sett_lag_behaviour = qcf.SettLagBehaviour.DONT_MOVE,
)

# Se da de alta el objeto
pata_bono = qcf.LegFactory.build_bullet_fixed_rate_leg(**params_bono)

In [ ]:
tera = qcf.QCInterestRate(.019071, qcf.QCAct365(), qcf.QCCompoundWf())
bono_chileno = qcf.ChileanFixedRateBond(pata_bono, tera)

In [ ]:
aux.leg_as_dataframe(pata_bono)

## Swap ICP a 5Y

Hay que dar de alta la pata fija (CLF) y la pata ICPCLP.

In [ ]:
ny = aux.get_business_calendar("US", range(2025, 2030))

In [ ]:
params_pata_fija_swap = dict(
    rec_pay = qcf.RecPay.PAY,
    start_date = (start_date := qcf.QCDate(30, 9, 2025)),
    end_date = (end_date := qcf.QCDate(1, 9, 2030)),
    bus_adj_rule = (bus_adj_rule := qcf.BusyAdjRules.MODFOLLOW),
    settlement_periodicity = (settlement_periodicity := qcf.Tenor('6M')),
    settlement_stub_period = (settlement_stub_period := qcf.StubPeriod.SHORTFRONT),
    settlement_calendar = (settlement_calendar := (scl + ny)),
    settlement_lag = (settlement_lag := 0),
    initial_notional = notional_uf,
    amort_is_cashflow = (amort_is_cashflow := True),
    interest_rate = qcf.QCInterestRate(
        # 0.016450,
        0.016385,
        qcf.QCAct360(),
        qcf.QCLinearWf(),
    ),
    notional_currency = qcf.QCCLF(),
    is_bond = False,
    sett_lag_behaviour = (sett_lag_behaviour := qcf.SettLagBehaviour.DONT_MOVE),
)

pata_fija_swap = qcf.LegFactory.build_bullet_fixed_rate_leg(**params_pata_fija_swap)

In [ ]:
aux.leg_as_dataframe(pata_fija_swap)

In [ ]:
params_pata_icpclp = dict(
    rec_pay=qcf.RecPay.RECEIVE,
    start_date=start_date,
    end_date=end_date,
    bus_adj_rule=bus_adj_rule,
    fix_adj_rule=qcf.BusyAdjRules.PREVIOUS,
    settlement_periodicity=settlement_periodicity,
    settlement_stub_period=settlement_stub_period,
    settlement_calendar=settlement_calendar,
    fixing_calendar=scl,
    settlement_lag=settlement_lag,
    initial_notional= (notional_clp:=notional_uf * 39_485.65),
    amort_is_cashflow=amort_is_cashflow,
    spread=0.0,
    gearing=1.0,
    interest_rate = qcf.QCInterestRate(
        0.0,
        qcf.QCAct360(),
        qcf.QCLinearWf(),
    ),
    index_name="ICPCLP",
    eq_rate_decimal_places=4,
    notional_currency=qcf.QCCLP(),
    dates_for_eq_rate=qcf.DatesForEquivalentRate.ACCRUAL,
    sett_lag_behaviour=sett_lag_behaviour,
)

pata_icpclp_swap = qcf.LegFactory.build_bullet_overnight_index_leg(**params_pata_icpclp)

## Valorización al 2025-09-26

In [ ]:
tir_mercado = {
    (fecha_hoy := qcf.QCDate("2025-09-26")): qcf.QCInterestRate(.022319, qcf.QCAct365(), qcf.QCCompoundWf()),
}

tir_mercado_up = {
    qcf.QCDate("2025-09-26"): qcf.QCInterestRate(.022419, qcf.QCAct365(), qcf.QCCompoundWf()),
}

In [ ]:
ufs = {
    fecha_hoy: 39_485.65,
    start_date: 39_485.65,
}

icp = {
    fecha_hoy: 25_713.89,
    start_date: 25_724.07,
}


In [ ]:
pv = qcf.PresentValue()
fwd = qcf.ForwardRates()

Valor presente y sensibilidad de toda la estructura.

In [ ]:
valor_presente_base = aux.vp_swap_spread(
    fecha_hoy,
    # Bono
    pata_bono,
    tir_mercado[fecha_hoy],
    tir_mercado[fecha_hoy],
    # Pata Fija Swap
    pata_fija_swap,
    qcf_cclfcolusd,
    # Pata ICPCLP Swap
    pata_icpclp_swap,
    qcf_cicpclp,
    qcf_cclpcolusd,
    ufs[fecha_hoy],
    icp[fecha_hoy],
    icp[fecha_hoy],
    pv,
    fwd,
)

In [ ]:
print("###### Bono ######")
print(f"Valor Presente: {valor_presente_base.pv_bono:,.0f}")
print(f"Sensibilidad: {valor_presente_base.sens_bono:,.0f}")
print(f"Aj. Valor Mercado: {valor_presente_base.ajuste_valor_mercado_bono:,.0f}")

print("\n###### Pata Fija Swap ######")
print(f"Valor Presente: {valor_presente_base.pv_pata_fija_swap:,.0f}")
print(f"K + Interés Devengado: {valor_presente_base.accrued_value_fija_swap:,.0f}")
print(f"Sensibilidad: {valor_presente_base.sens_pata_fija_swap:,.0f}")

print("\n###### Pata ICPCLP Swap ######")
print(f"Valor Presente: {valor_presente_base.pv_pata_icpclp:,.0f}")
print(f"K + Interés Devengado: {valor_presente_base.accrued_value_icpclp_swap:,.0f}")
print(f"Sensibilidad Proyección: {valor_presente_base.sens_proyeccion_pata_icpclp_swap:,.0f}")
print(f"Sensibilidad Descuento: {valor_presente_base.sens_descuento_pata_icpclp_swap:,.0f}")

print("\n###### Todo el Swap ######")
print(f"Valor Presente: {valor_presente_base.pv_pata_fija_swap + valor_presente_base.pv_pata_icpclp:,.0f}")
print(f"Sensibilidad: {valor_presente_base.sens_pata_fija_swap +
                       valor_presente_base.sens_proyeccion_pata_icpclp_swap +
                       valor_presente_base.sens_descuento_pata_icpclp_swap:,.0f}")

print(f"\nMark to Market Total: {valor_presente_base.valor_presente_total:,.0f}")
aj_total = (
        valor_presente_base.ajuste_valor_mercado_bono +
        valor_presente_base.pv_pata_fija_swap - valor_presente_base.accrued_value_fija_swap +
        valor_presente_base.pv_pata_icpclp - valor_presente_base.accrued_value_icpclp_swap
)
print(f"Ajuste a Valor de Mercado Total: {aj_total:,.0f}")

## Cálculo de Swap Spread

In [ ]:
pv_bono_swap = pv.pv(fecha_hoy, pata_bono, qcf_cclfcolusd) * ufs[fecha_hoy]
print(f"PV Bono con Swap: {pv_bono_swap:,.0f}")

In [ ]:
tir_swap = aux.encuentra_tir_swap(fecha_hoy, pata_bono, 0.019, pv_bono_swap / ufs[fecha_hoy], pv)
print(f"tir_swap: {tir_swap:,.4%}")
print(f"Swap Spread: {(swap_spread := tir_mercado[fecha_hoy].get_value() - tir_swap):,.4%}")

In [ ]:
bp = 1
swap_spread_up = swap_spread + bp / 10_000.0
print(f"Swap Spread + {bp} pb: {swap_spread_up:.4%}")
tir_mercado_up = qcf.QCInterestRate(tir_swap + swap_spread_up, qcf.QCAct365(), qcf.QCCompoundWf())

In [ ]:
valor_presente_up = aux.vp_swap_spread(
    fecha_hoy,
    # Bono
    pata_bono,
    tir_mercado_up,
    tir_mercado[fecha_hoy],
    # Pata Fija Swap
    pata_fija_swap,
    qcf_cclfcolusd,
    # Pata ICPCLP Swap
    pata_icpclp_swap,
    qcf_cicpclp,
    qcf_cclpcolusd,
    ufs[fecha_hoy],
    icp[fecha_hoy],
    icp[fecha_hoy],
    pv,
    fwd,
)

In [ ]:
print(f"PV Bono: {valor_presente_up.pv_bono:,.0f}")
print(f"Sens Bono: {valor_presente_up.sens_bono:,.0f}")
print(f"\nPV Pata Fija Swap: {valor_presente_up.pv_pata_fija_swap:,.0f}")
print(f"Sens Pata Fija Swap: {valor_presente_up.sens_pata_fija_swap:,.0f}")
print(f"\nPV Pata ICPCLP: {valor_presente_up.pv_pata_icpclp:,.0f}")
print(f"Sens Proyección Pata ICPCLP: {valor_presente_up.sens_proyeccion_pata_icpclp_swap:,.0f}")
print(f"Sens Descuento ICPCLP: {valor_presente_up.sens_descuento_pata_icpclp_swap:,.0f}")
print(f"\nMark to Market Total: {valor_presente_up.valor_presente_total:,.0f}")
print(f"Delta: {valor_presente_up.valor_presente_total - valor_presente_base.valor_presente_total:,.0f}")

In [ ]:
swap_spread_down = swap_spread - bp / 10_000.0
print(f"Swap Spread - {bp} pb: {swap_spread_down:.4%}")
tir_mercado_down = qcf.QCInterestRate(tir_swap + swap_spread_down, qcf.QCAct365(), qcf.QCCompoundWf())

In [ ]:
valor_presente_down = aux.vp_swap_spread(
    fecha_hoy,
    # Bono
    pata_bono,
    tir_mercado_down,
    tir_mercado[fecha_hoy],
    # Pata Fija Swap
    pata_fija_swap,
    qcf_cclfcolusd,
    # Pata ICPCLP Swap
    pata_icpclp_swap,
    qcf_cicpclp,
    qcf_cclpcolusd,
    ufs[fecha_hoy],
    icp[fecha_hoy],
    icp[fecha_hoy],
    pv,
    fwd,
)

In [ ]:
print(f"PV Bono: {valor_presente_down.pv_bono:,.0f}")
print(f"Sens Bono: {valor_presente_down.sens_bono:,.0f}")
print(f"\nPV Pata Fija Swap: {valor_presente_down.pv_pata_fija_swap:,.0f}")
print(f"Sens Pata Fija Swap: {valor_presente_down.sens_pata_fija_swap:,.0f}")
print(f"\nPV Pata ICPCLP: {valor_presente_down.pv_pata_icpclp:,.0f}")
print(f"Sens Proyección Pata ICPCLP: {valor_presente_down.sens_proyeccion_pata_icpclp_swap:,.0f}")
print(f"Sens Descuento ICPCLP: {valor_presente_down.sens_descuento_pata_icpclp_swap:,.0f}")
print(f"\nMark to Market Total: {valor_presente_down.valor_presente_total:,.0f}")
print(f"Delta: {valor_presente_down.valor_presente_total - valor_presente_base.valor_presente_total:,.0f}")

## Escenarios de Curvas

In [ ]:
qcf_cicpclp_1 = aux.build_zero_coupon_curve(
    df_cicpclp.maturity.values,
    df_cicpclp.rate_1.values,
    cyf := qcw.YearFraction.ACT365,
    cwf := qcw.WealthFactor.CON,
)

In [ ]:
valor_presente_esc_1 = aux.vp_swap_spread(
    fecha_hoy,
    # Bono
    pata_bono,
    tir_mercado[fecha_hoy],
    tir_mercado[fecha_hoy],
    # Pata Fija Swap
    pata_fija_swap,
    qcf_cclfcolusd,
    # Pata ICPCLP Swap
    pata_icpclp_swap,
    qcf_cicpclp_1,
    qcf_cclpcolusd,
    ufs[fecha_hoy],
    icp[fecha_hoy],
    icp[fecha_hoy],
    pv,
    fwd,
)

In [ ]:
print(f"PV Bono: {valor_presente_esc_1.pv_bono:,.0f}")
print(f"Sens Bono: {valor_presente_esc_1.sens_bono:,.0f}")
print(f"\nPV Pata Fija Swap: {valor_presente_esc_1.pv_pata_fija_swap:,.0f}")
print(f"Sens Pata Fija Swap: {valor_presente_esc_1.sens_pata_fija_swap:,.0f}")
print(f"\nPV Pata ICPCLP: {valor_presente_esc_1.pv_pata_icpclp:,.0f}")
print(f"Sens Proyección Pata ICPCLP: {valor_presente_esc_1.sens_proyeccion_pata_icpclp_swap:,.0f}")
print(f"Sens Descuento ICPCLP: {valor_presente_esc_1.sens_descuento_pata_icpclp_swap:,.0f}")
print(f"\nMark to Market Total: {valor_presente_esc_1.valor_presente_total:,.0f}")
print(f"Delta: {valor_presente_esc_1.valor_presente_total - valor_presente_base.valor_presente_total:,.0f}")

## Escenarios de Curva y Spread Después de 1 Mes

In [ ]:
fecha_1_mes = qcf.QCDate("2025-10-27")
uf_1_mes = 39_577.28
icp_1_mes = icp[fecha_hoy] * (1 + .0475 * 32 / 360)
tir_compra = tir_mercado[fecha_hoy].get_value()

### Sube Curva UF mantiene Spread

In [ ]:
bp = 100
tir_mercado = aux.make_tir(tir_swap + bp / 10_000.0 + swap_spread)

In [ ]:
qcf_cclfcolusd_1 = aux.build_zero_coupon_curve(
    df_cclfcolusd.maturity.values,
    df_cclfcolusd.rate_1.values,
    cyf := qcw.YearFraction.ACT365,
    cwf := qcw.WealthFactor.CON,
)

In [ ]:
sube_curva_igual_spread = aux.vp_swap_spread(
    fecha_1_mes,
    # Bono
    pata_bono,
    tir_mercado,
    aux.make_tir(tir_compra),
    # Pata Fija Swap
    pata_fija_swap,
    qcf_cclfcolusd_1,
    # Pata ICPCLP Swap
    pata_icpclp_swap,
    qcf_cicpclp,
    qcf_cclpcolusd,
    uf_1_mes,
    icp[start_date],
    icp_1_mes,
    pv,
    fwd,
)

In [ ]:
fff = "{:,.0f}"
format_dict = {
    "pv_bono": fff,
}

In [ ]:
ForDF = namedtuple('ForDF', [
    'pv_bono',
    'pv_compra_bono',
    'ajuste_valor_mercado_bono',
    'pv_pata_fija_swap',
    'accrued_value_fija_swap',
    'ajuste_valor_mercado_fija_swap',
    'pv_pata_icpclp',
    'accrued_value_icpclp_swap',
    'ajuste_valor_mercado_icpclp_swap',
    'valor_presente_total',
])

In [ ]:
for_df = ForDF(
    pv_bono=sube_curva_igual_spread.pv_bono,
    pv_compra_bono=sube_curva_igual_spread.pv_compra_bono,
    ajuste_valor_mercado_bono=sube_curva_igual_spread.ajuste_valor_mercado_bono,
    pv_pata_fija_swap=(pv_mkt_fija_swap:=sube_curva_igual_spread.pv_pata_fija_swap),
    accrued_value_fija_swap=(accrued_fija_swap:=sube_curva_igual_spread.accrued_value_fija_swap),
    ajuste_valor_mercado_fija_swap=pv_mkt_fija_swap - accrued_fija_swap,
    pv_pata_icpclp=(pv_mkt_icpclp_swap:=sube_curva_igual_spread.pv_pata_icpclp),
    accrued_value_icpclp_swap=(accrued_icpclp_swap:=sube_curva_igual_spread.accrued_value_icpclp_swap),
    ajuste_valor_mercado_icpclp_swap=pv_mkt_icpclp_swap - accrued_icpclp_swap,
    valor_presente_total=sube_curva_igual_spread.valor_presente_total,
)

In [ ]:
pd.DataFrame([for_df]).style.format(fff)

In [ ]:
devengo_bono = sube_curva_igual_spread.pv_compra_bono - valor_presente_base.pv_compra_bono
print(f"Devengo Bono: {devengo_bono:,.0f}")

In [ ]:
devengo_fija_swap = sube_curva_igual_spread.accrued_value_fija_swap - valor_presente_base.accrued_value_fija_swap
devengo_icpclp_swap = sube_curva_igual_spread.accrued_value_icpclp_swap - valor_presente_base.accrued_value_icpclp_swap
print(f"Devengo Swap: {devengo_fija_swap + devengo_icpclp_swap:,.0f}")

In [ ]:
ajuste_bono = sube_curva_igual_spread.ajuste_valor_mercado_bono
print(f"Ajuste a Valor de Mercado Bono: {ajuste_bono:,.0f}")

In [ ]:
ajuste_fija_swap = sube_curva_igual_spread.pv_pata_fija_swap - sube_curva_igual_spread.accrued_value_fija_swap
print(f"Ajuste a Valor de Mercado Fija Swap: {ajuste_fija_swap:,.0f}")

In [ ]:
ajuste_icpclp_swap = sube_curva_igual_spread.pv_pata_icpclp - sube_curva_igual_spread.accrued_value_icpclp_swap
print(f"Ajuste a Valor de Mercado ICPCLP Swap: {ajuste_icpclp_swap:,.0f}")

In [ ]:
print(f"Ajuste a Valor de Mercado Swap: {ajuste_fija_swap + ajuste_icpclp_swap:,.0f}")

### Mantiene Curva UF Sube Spread

In [ ]:
bp = 100
tir_mercado = aux.make_tir(tir_swap + bp / 10_000.0 + swap_spread)

In [ ]:
igual_curva_sube_spread = aux.vp_swap_spread(
    fecha_1_mes,
    # Bono
    pata_bono,
    tir_mercado,
    aux.make_tir(tir_compra),
    # Pata Fija Swap
    pata_fija_swap,
    qcf_cclfcolusd,
    # Pata ICPCLP Swap
    pata_icpclp_swap,
    qcf_cicpclp,
    qcf_cclpcolusd,
    uf_1_mes,
    icp[start_date],
    icp_1_mes,
    pv,
    fwd,
)

In [ ]:
for_df = ForDF(
    pv_bono=igual_curva_sube_spread.pv_bono,
    pv_compra_bono=igual_curva_sube_spread.pv_compra_bono,
    ajuste_valor_mercado_bono=igual_curva_sube_spread.ajuste_valor_mercado_bono,
    pv_pata_fija_swap=(pv_mkt_fija_swap:=igual_curva_sube_spread.pv_pata_fija_swap),
    accrued_value_fija_swap=(accrued_fija_swap:=igual_curva_sube_spread.accrued_value_fija_swap),
    ajuste_valor_mercado_fija_swap=pv_mkt_fija_swap - accrued_fija_swap,
    pv_pata_icpclp=(pv_mkt_icpclp_swap:=igual_curva_sube_spread.pv_pata_icpclp),
    accrued_value_icpclp_swap=(accrued_icpclp_swap:=igual_curva_sube_spread.accrued_value_icpclp_swap),
    ajuste_valor_mercado_icpclp_swap=pv_mkt_icpclp_swap - accrued_icpclp_swap,
    valor_presente_total=igual_curva_sube_spread.valor_presente_total,
)

In [ ]:
pd.DataFrame([for_df]).style.format(fff)

In [ ]:
devengo_bono = igual_curva_sube_spread.pv_compra_bono - valor_presente_base.pv_compra_bono
print(f"Devengo Bono: {devengo_bono:,.0f}")

In [ ]:
devengo_fija_swap = igual_curva_sube_spread.accrued_value_fija_swap - valor_presente_base.accrued_value_fija_swap
devengo_icpclp_swap = igual_curva_sube_spread.accrued_value_icpclp_swap - valor_presente_base.accrued_value_icpclp_swap
print(f"Devengo Swap: {devengo_fija_swap + devengo_icpclp_swap:,.0f}")

In [ ]:
ajuste_bono = igual_curva_sube_spread.ajuste_valor_mercado_bono
print(f"Ajuste a Valor de Mercado Bono: {ajuste_bono:,.0f}")

In [ ]:
ajuste_fija_swap = igual_curva_sube_spread.pv_pata_fija_swap - igual_curva_sube_spread.accrued_value_fija_swap
print(f"Ajuste a Valor de Mercado Fija Swap: {ajuste_fija_swap:,.0f}")

In [ ]:
ajuste_icpclp_swap = igual_curva_sube_spread.pv_pata_icpclp - igual_curva_sube_spread.accrued_value_icpclp_swap
print(f"Ajuste a Valor de Mercado ICPCLP Swap: {ajuste_icpclp_swap:,.0f}")

In [ ]:
print(f"Ajuste a Valor de Mercado Swap: {ajuste_fija_swap + ajuste_icpclp_swap:,.0f}")